In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

current = Path.cwd()
project_root = current.parent.parent
sys.path.append(str(project_root))

from utils import add_project_root_to_sys_path
add_project_root_to_sys_path()

In [41]:
from data.linear import LinearTaskGenerator, TaskGenerator
from models.linear_attention import SingleLayerLSA
from experiments.single_layer.train import train_ICL_model
from experiments.single_layer.test import compare_SignelLSA_matrices
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.optim as optim

### Implicit weight extraction

In [ ]:
NX = 10
NY = 1

N = 100  # Important increase
W_STD = 1.0
X_RANGE = (-1, 1)

N_STEPS = 1000
BATCH_SIZE = 2048
LR = 0.001

ETA_VALUES = np.logspace(-3, 2, 100)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
generator = LinearTaskGenerator(nx=NX, ny=NY, x_range=X_RANGE, w_std=W_STD)
model = SingleLayerLSA(nx=NX, ny=NY)
optimizer = optim.AdamW(model.parameters(), lr=LR)
device

'cpu'

In [ ]:
model, losses, tf_val, gd_val, best_eta = train_ICL_model(
    model=model,
    task_generator=generator,
    optimizer=optimizer,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    lr=LR,
    N=N,
    device=device,
    verbose=True,
    test_interval=100,
    val_tasks=1000,
    eta_values=ETA_VALUES
)

In [53]:
from sklearn.linear_model import LinearRegression

def weight_extraction(model,
                      samples_num: int = 1000,
                      N: int = N,
                      NX: int = NX, NY: int = NY,
                      generator: TaskGenerator = generator,
                      refresh_generator: bool = False,
                      verbose: bool = True
                      ):
    """
    TODO
    
    Args:
        model: модель, реализующая методы forward и predict
        task_generator: Генератор задач с методом .generate_task(N).
        N: Количество обучающих примеров в контексте.
        NX, NY: Размерности задачи
        verbose: Если True, показывает прогресс-бар и графики.
    
    Returns:
        Matrix: Наилучше приближающая модель матрица
        R^2: R^2-score матрицы
        
    """
    tokens, targets = [], []
    assert samples_num > 20 * (NX + NY), "Количество сэмплов должно быть сильно больше размерности векторов"

    if refresh_generator:
        generator.generate_teacher()
    
    for _ in range(samples_num):
        x, y_true = generator.generate_task(N=N, new_params=False)
        if len(tokens) != len(tokens):
            tokens.append(tokens[-1])
            tokens[-1][N] = x[N]
            targets.append(y_true)
            continue
        tokens.append(x)
        targets.append(y_true)
        
    tokens = torch.FloatTensor(np.array(tokens)).to(device)
    tokens_ends = tokens.detach().numpy()[:, N, :NX]
    targets = np.array(targets)

    pred = model.predict(tokens).detach().numpy()

    W_true = generator.teacher_params
    lin_reg = LinearRegression().fit(tokens_ends, pred)

    if verbose:
        print("||W_true - W_extr||_2 =", np.linalg.norm(W_true - lin_reg.coef_))
        print("R^2 =", lin_reg.score(tokens_ends, pred))
    return lin_reg.coef_, lin_reg.score(tokens_ends, pred)

print(weight_extraction(model, generator=generator, refresh_generator=True)[0])
print(generator.teacher_params)

||W_true - W_extr||_2 = 0.4911390801456076
R^2 = 0.9140836596488953
[[-0.4325819   1.5565252   2.1724572  -1.1778053  -0.13082471 -0.33697823
   2.0061705  -1.0968257  -0.9459068  -0.08518708]]
[[-5.05431353e-01  1.73551831e+00  2.30235508e+00 -1.34187360e+00
  -1.29455851e-01 -5.07885136e-01  2.26481804e+00 -1.28053502e+00
  -1.09737345e+00  1.31941449e-04]]


## Notes

1. Замораживаем модель

2. Генерируем X (сильно больще из размерности)
 ? как генерировать ? 
 ! генерируем как X на обучении и на тестах !
 
3. Фиксируем контекст 
 ? какой выбрать контекст ? : Если не фиксированная длина, то нетривиальный выбор размераx
 
4. Получаем предсказания модели Y^ (теперь |X| = |Y^|)

5. Обучаем матрицу размера  d_x * d_y оптимизируя некоторую метрику (наверное, MSE) для предсказания Y^ по X.

6. Считаем статистику верхней линейной регрессии, насколько она валидна (statsmodels). 

Псевдокод:
```
Matrix (SampleSIZE, d_x, d_y, TrueTheta, GenerateX, Model, Noise) 
{
    X = GenerateX(SampleSIZE, d_x)
    Y = Noise(X^T @ TrueTheta)
    
    PointSIZE = 50 * (d_x + d_y)  # чтобы было гарантировано больше чем размерность
    
    X' = GenerateX(PointSIZE, d_x)
    Y'_i = Model(
        Context: (X, Y) + {X'_i, 0},
    )  # получаем пары (X' Y') те, что предсказал трансформер
    
    EstTheta = LinReg(X', Y')  # неявная матрица
    
    return {EstTheta, check(X', Y', EstTheta)}   # выводим матрицу и R^2
}
(EstTheta, Coef.determ)
```